# BTCUSDT Research Collector — FINAL

**One-click, RAM-safe, restartable, source-aware collector.**

Run **Runtime → Run all**.

This version handles Binance archive frequencies correctly, validates ZIPs/checksums, avoids blind 404 retries, streams aggTrades in chunks, and refuses to create a false OI-free dataset.

In [ ]:
!pip -q install pandas==2.2.3 requests==2.32.4 pyarrow==18.1.0 tqdm python-dateutil
print('Pinned dependencies installed.')


In [ ]:
from pathlib import Path
import os, re, io, json, time, zipfile, hashlib, shutil, math
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import requests

# USER CONFIGURATION
START_DATE=os.environ.get('START_DATE','2025-01-01')
END_DATE=os.environ.get('END_DATE','2026-05-31')
SYMBOL='BTCUSDT'
MODE=os.environ.get('MODE','CORE').upper()
MAX_WORKERS=10
CHUNK_ROWS=500000
TIMEOUT=120
ROOT=Path('/content/btcusdt_research_final')
CACHE=ROOT/'cache'; OUT=ROOT/'output'; FLOW=OUT/'flow_5m'; REPORTS=OUT/'reports'
for p in (CACHE,OUT,FLOW,REPORTS): p.mkdir(parents=True,exist_ok=True)
START=pd.Timestamp(START_DATE,tz='UTC'); END=pd.Timestamp(END_DATE,tz='UTC')
FUT_BASE='https://data.binance.vision/data/futures/um'; SPOT_BASE='https://data.binance.vision/data/spot'
print('MODE:',MODE,'PERIOD:',START,'->',END)


In [ ]:
def make_session():
    s=requests.Session(); s.headers.update({'User-Agent':'BTCUSDT-research-collector-final/1.0','Accept-Encoding':'gzip, deflate'}); return s

def sha256_file(path,chunk_size=8*1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(chunk_size),b''): h.update(b)
    return h.hexdigest()

def checksum_for(url,session):
    try:
        r=session.get(url+'.CHECKSUM',timeout=30)
        if r.status_code!=200: return None
        m=re.search(r'\b([0-9a-fA-F]{64})\b',r.text)
        return m.group(1).lower() if m else None
    except Exception: return None

def download(url,target,retries=6):
    target=Path(target); target.parent.mkdir(parents=True,exist_ok=True); part=Path(str(target)+'.part'); s=make_session()
    for attempt in range(1,retries+1):
        try:
            existing=part.stat().st_size if part.exists() else 0
            headers={'Range':f'bytes={existing}-'} if existing else {}
            with s.get(url,headers=headers,stream=True,timeout=TIMEOUT) as r:
                if r.status_code==404: return {'status':'not_found','url':url,'path':str(target)}
                if existing and r.status_code==206: mode='ab'
                elif existing and r.status_code==200: existing=0; mode='wb'
                elif not existing and r.status_code==200: mode='wb'
                else: r.raise_for_status(); mode='wb'
                r.raise_for_status()
                with open(part,mode) as f:
                    for chunk in r.iter_content(chunk_size=1024*1024):
                        if chunk: f.write(chunk)
            part.replace(target)
            with zipfile.ZipFile(target,'r') as z:
                if z.testzip() is not None: raise RuntimeError('ZIP integrity test failed')
            expected=checksum_for(url,s); actual=sha256_file(target)
            if expected and actual!=expected: raise RuntimeError('SHA256 checksum mismatch')
            return {'status':'ok','url':url,'path':str(target),'sha256':actual,'size_bytes':target.stat().st_size}
        except Exception as e:
            if attempt==retries: return {'status':'failed','url':url,'path':str(target),'error':str(e)}
            time.sleep(min(2**(attempt-1),20))
    return {'status':'failed','url':url,'path':str(target)}

def month_starts(a,b):
    x=a.normalize().replace(day=1); out=[]
    while x<=b: out.append(x); x=x+pd.offsets.MonthBegin(1)
    return out

def day_starts(a,b):
    x=a.normalize(); out=[]
    while x<=b: out.append(x); x=x+pd.Timedelta(days=1)
    return out

def monthly_job(market,dataset,interval,m):
    stamp=f'{m.year:04d}-{m.month:02d}'; base=FUT_BASE if market=='futures' else SPOT_BASE
    if dataset=='klines': url=f'{base}/monthly/klines/{SYMBOL}/{interval}/{SYMBOL}-{interval}-{stamp}.zip'; target=CACHE/market/dataset/interval/f'{stamp}.zip'
    elif dataset=='aggTrades': url=f'{base}/monthly/aggTrades/{SYMBOL}/{SYMBOL}-aggTrades-{stamp}.zip'; target=CACHE/market/dataset/f'{stamp}.zip'
    elif dataset=='fundingRate': url=f'{base}/monthly/fundingRate/{SYMBOL}/{SYMBOL}-fundingRate-{stamp}.zip'; target=CACHE/market/dataset/f'{stamp}.zip'
    else: raise ValueError(dataset)
    return (market,dataset,interval,stamp,url,target)

def daily_job(market,dataset,d):
    stamp=f'{d.year:04d}-{d.month:02d}-{d.day:02d}'; base=FUT_BASE if market=='futures' else SPOT_BASE
    url=f'{base}/daily/{dataset}/{SYMBOL}/{SYMBOL}-{dataset}-{stamp}.zip'; target=CACHE/market/dataset/f'{stamp}.zip'
    return (market,dataset,None,stamp,url,target)

def build_jobs():
    jobs=[]
    for m in month_starts(START,END):
        jobs += [monthly_job('futures','klines','5m',m), monthly_job('spot','klines','5m',m), monthly_job('futures','aggTrades',None,m), monthly_job('futures','fundingRate',None,m)]
    for d in day_starts(START,END): jobs.append(daily_job('futures','metrics',d))
    if MODE=='MICROSTRUCTURE':
        for d in day_starts(START,END): jobs.append(daily_job('futures','bookDepth',d))
    return jobs

def download_job(job):
    market,dataset,interval,period,url,target=job
    if target.exists() and target.stat().st_size>100:
        try:
            with zipfile.ZipFile(target,'r') as z:
                if z.testzip() is not None: raise RuntimeError('cached ZIP corrupt')
            actual=sha256_file(target); expected=checksum_for(url,make_session())
            if not expected or expected==actual:
                return {'market':market,'dataset':dataset,'interval':interval,'period':period,'url':url,'path':str(target),'status':'cached','sha256':actual,'size_bytes':target.stat().st_size}
        except Exception: target.unlink(missing_ok=True)
    rec=download(url,target); rec.update({'market':market,'dataset':dataset,'interval':interval,'period':period}); return rec

jobs=build_jobs(); print('Planned archive objects:',len(jobs))
results=[]
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futs=[ex.submit(download_job,j) for j in jobs]
    for i,f in enumerate(as_completed(futs),1):
        rec=f.result(); results.append(rec)
        if i%25==0 or rec['status'] not in ('ok','cached'): print(i,rec['dataset'],rec['period'],rec['status'])
Path(REPORTS/'download_manifest.json').write_text(json.dumps(results,indent=2,default=str))
from collections import Counter
print('DOWNLOAD SUMMARY:',dict(Counter(r['status'] for r in results)))


In [ ]:
def successful_records(dataset=None,market=None):
    x=[]
    for r in results:
        if r['status'] not in ('ok','cached'): continue
        if dataset and r['dataset']!=dataset: continue
        if market and r['market']!=market: continue
        x.append(r)
    return sorted(x,key=lambda r:r['period'])

def csv_members(zp):
    with zipfile.ZipFile(zp,'r') as z: return [n for n in z.namelist() if n.lower().endswith('.csv')]

def read_csv_from_zip(zp,member,**kwargs):
    with zipfile.ZipFile(zp,'r') as z:
        with z.open(member) as f: return pd.read_csv(f,**kwargs)

def load_klines(records):
    frames=[]; audit=[]
    for rec in records:
        try:
            members=csv_members(rec['path']); rows=0
            for member in members:
                d=read_csv_from_zip(rec['path'],member,header=None,low_memory=False)
                if d.shape[1]<12: continue
                d=d.iloc[:,:12].copy(); d.columns=['open_ms','open','high','low','close','volume','close_ms','quote_volume','trade_count','taker_buy_base','taker_buy_quote','ignore']
                d['ts']=pd.to_datetime(pd.to_numeric(d['open_ms'],errors='coerce'),unit='ms',utc=True,errors='coerce')
                for c in ['open','high','low','close','volume','quote_volume','trade_count','taker_buy_base','taker_buy_quote']: d[c]=pd.to_numeric(d[c],errors='coerce')
                d=d.dropna(subset=['ts','close']); rows += len(d)
                frames.append(d[['ts','open','high','low','close','volume','quote_volume','trade_count','taker_buy_base','taker_buy_quote']])
            audit.append({**rec,'rows':rows,'parse_status':'ok' if rows else 'empty'})
        except Exception as e: audit.append({**rec,'rows':0,'parse_status':'failed','error':str(e)})
    return (pd.concat(frames,ignore_index=True) if frames else pd.DataFrame(),audit)

fut_k,fut_audit=load_klines(successful_records('klines','futures'))
spot_k,spot_audit=load_klines(successful_records('klines','spot'))
if fut_k.empty: raise RuntimeError('FATAL: no usable FUTURES 5m data')
if spot_k.empty: raise RuntimeError('FATAL: no usable SPOT 5m data')
def clean(d): return d[(d.ts>=START)&(d.ts<=END)].sort_values('ts').drop_duplicates('ts').reset_index(drop=True)
fut_k=clean(fut_k); spot_k=clean(spot_k)
print('Futures rows:',len(fut_k),'Spot rows:',len(spot_k))


In [ ]:
# AGGTRADES -> 5m FLOW, streamed in chunks
def boolify(x):
    s=str(x).strip().lower()
    return x if isinstance(x,bool) else s in ('true','1','yes')

def aggregate_aggtrade_archive(zp):
    pieces=[]
    with zipfile.ZipFile(zp,'r') as z:
        for member in [n for n in z.namelist() if n.lower().endswith('.csv')]:
            with z.open(member) as f:
                reader=pd.read_csv(f,header=None,usecols=range(7),names=['id','price','qty','first_id','last_id','ms','buyer_maker'],chunksize=CHUNK_ROWS,low_memory=False)
                for d in reader:
                    d['ms']=pd.to_numeric(d['ms'],errors='coerce'); d['price']=pd.to_numeric(d['price'],errors='coerce'); d['qty']=pd.to_numeric(d['qty'],errors='coerce'); d=d.dropna(subset=['ms','price','qty'])
                    bm=d['buyer_maker'].map(boolify); d['signed_qty']=d['qty'].where(~bm,-d['qty']); d['signed_notional']=d['signed_qty']*d['price']; d['notional']=d['qty']*d['price']; d['bucket']=pd.to_datetime(d['ms'],unit='ms',utc=True).dt.floor('5min')
                    pieces.append(d.groupby('bucket').agg(agg_delta_base=('signed_qty','sum'),agg_delta_usdt=('signed_notional','sum'),agg_volume_usdt=('notional','sum'),agg_trades=('id','size')).reset_index())
    if not pieces: return pd.DataFrame()
    x=pd.concat(pieces,ignore_index=True).groupby('bucket',as_index=False).sum(numeric_only=True).rename(columns={'bucket':'ts'})
    return x

flow_parts=[]; flow_audit=[]
for rec in successful_records('aggTrades','futures'):
    try:
        x=aggregate_aggtrade_archive(Path(rec['path'])); x=x[(x.ts>=START)&(x.ts<=END)]
        flow_parts.append(x); flow_audit.append({**rec,'rows_5m':len(x),'parse_status':'ok' if len(x) else 'empty'})
    except Exception as e: flow_audit.append({**rec,'rows_5m':0,'parse_status':'failed','error':str(e)})
flow=pd.concat(flow_parts,ignore_index=True).groupby('ts',as_index=False).sum(numeric_only=True) if flow_parts else pd.DataFrame()
if flow.empty: raise RuntimeError('FATAL: no aggTrade flow could be reconstructed')
flow.to_parquet(FLOW/'futures_agg_flow_5m.parquet',index=False,compression='zstd')
print('5m agg flow rows:',len(flow))


In [ ]:
# METRICS / OI / CROWDING
def parse_metric_time(s):
    ss=s.astype('string'); n=pd.to_numeric(ss,errors='coerce'); out=pd.Series(pd.NaT,index=s.index,dtype='datetime64[ns, UTC]')
    masks=[('ms',n.notna()&(n.abs()>=10**11)&(n.abs()<10**14)),('us',n.notna()&(n.abs()>=10**14)),('s',n.notna()&(n.abs()<10**11))]
    for unit,mask in masks:
        if mask.any(): out.loc[mask]=pd.to_datetime(n.loc[mask],unit=unit,utc=True,errors='coerce')
    mask=out.isna()
    if mask.any(): out.loc[mask]=pd.to_datetime(ss.loc[mask],utc=True,errors='coerce')
    return out

metric_frames=[]; metric_audit=[]
for rec in successful_records('metrics','futures'):
    try:
        rows=0
        for member in csv_members(rec['path']):
            d=read_csv_from_zip(rec['path'],member,low_memory=False); d.columns=[str(c).strip() for c in d.columns]
            tc=next((c for c in d.columns if c.lower()=='create_time'),None)
            if tc is None: continue
            d['ts']=parse_metric_time(d[tc]); ren={}
            for c in d.columns:
                q=str(c).lower().strip()
                if q=='sum_open_interest': ren[c]='oi_btc'
                elif q=='sum_open_interest_value': ren[c]='oi_value_usdt'
                elif 'count_toptrader_long_short_ratio' in q: ren[c]='top_ls_count_ratio'
                elif 'sum_toptrader_long_short_ratio' in q: ren[c]='top_ls_position_ratio'
                elif 'count_long_short_ratio' in q: ren[c]='global_ls_count_ratio'
                elif 'sum_taker_long_short_vol_ratio' in q: ren[c]='taker_ls_volume_ratio'
            d=d.rename(columns=ren); keep=['ts']+[c for c in ['oi_btc','oi_value_usdt','top_ls_count_ratio','top_ls_position_ratio','global_ls_count_ratio','taker_ls_volume_ratio'] if c in d.columns]; d=d[keep].dropna(subset=['ts'])
            for c in keep[1:]: d[c]=pd.to_numeric(d[c],errors='coerce')
            rows += len(d); metric_frames.append(d)
        metric_audit.append({**rec,'rows':rows,'parse_status':'ok' if rows else 'empty'})
    except Exception as e: metric_audit.append({**rec,'rows':0,'parse_status':'failed','error':str(e)})
if not metric_frames: raise RuntimeError('FATAL: no metrics rows parsed')
metrics=pd.concat(metric_frames,ignore_index=True).sort_values('ts').drop_duplicates('ts'); metrics=metrics[(metrics.ts>=START)&(metrics.ts<=END)].reset_index(drop=True)
if metrics['oi_btc'].notna().sum()==0: raise RuntimeError('FATAL: OI is empty')
print('Metrics rows:',len(metrics),'OI rows:',int(metrics.oi_btc.notna().sum()))


In [ ]:
# FUNDING
fund_parts=[]
for rec in successful_records('fundingRate','futures'):
    try:
        for member in csv_members(rec['path']):
            d=read_csv_from_zip(rec['path'],member,low_memory=False); d.columns=[str(c).strip() for c in d.columns]
            tcol=next((c for c in d.columns if c.lower() in ('calc_time','fundingtime','funding_time')),None); rcol=next((c for c in d.columns if 'fundingrate' in str(c).lower()),None)
            if not tcol or not rcol: continue
            ts=pd.to_datetime(pd.to_numeric(d[tcol],errors='coerce'),unit='ms',utc=True,errors='coerce'); mask=ts.isna(); ts.loc[mask]=pd.to_datetime(d.loc[mask,tcol],utc=True,errors='coerce')
            fund_parts.append(pd.DataFrame({'ts':ts,'funding_rate':pd.to_numeric(d[rcol],errors='coerce')}).dropna(subset=['ts','funding_rate']))
    except Exception as e: print('funding parse warning',e)
funding=pd.concat(fund_parts,ignore_index=True).sort_values('ts').drop_duplicates('ts') if fund_parts else pd.DataFrame()
if not funding.empty: funding=funding[(funding.ts>=START)&(funding.ts<=END)]
print('Funding rows:',len(funding))


In [ ]:
# OPTIONAL daily bookDepth snapshot proxy
depth=pd.DataFrame(); depth_audit=[]
if MODE=='MICROSTRUCTURE':
    parts=[]
    for rec in successful_records('bookDepth','futures'):
        try:
            for member in csv_members(rec['path']):
                d=read_csv_from_zip(rec['path'],member,low_memory=False); d.columns=[str(c).strip().lower() for c in d.columns]
                tcol=next((c for c in d.columns if c in ('timestamp','time','ts')),None); pcol=next((c for c in d.columns if c=='percentage'),None); dcol=next((c for c in d.columns if c=='depth'),None)
                if not all((tcol,pcol,dcol)): continue
                x=pd.DataFrame({'ts':pd.to_datetime(d[tcol],utc=True,errors='coerce'),'percentage':pd.to_numeric(d[pcol],errors='coerce'),'depth':pd.to_numeric(d[dcol],errors='coerce')}).dropna()
                bid=x[x.percentage<=-1].groupby('ts').depth.max().rename('depth_bid_1pct'); ask=x[x.percentage>=1].groupby('ts').depth.max().rename('depth_ask_1pct'); z=pd.concat([bid,ask],axis=1); z['depth_imbalance_1pct']=(z.depth_bid_1pct-z.depth_ask_1pct)/(z.depth_bid_1pct+z.depth_ask_1pct).replace(0,pd.NA); parts.append(z.reset_index())
            depth_audit.append({**rec,'parse_status':'ok'})
        except Exception as e: depth_audit.append({**rec,'parse_status':'failed','error':str(e)})
    if parts:
        depth=pd.concat(parts,ignore_index=True); depth['bucket']=depth.ts.dt.floor('5min'); depth=depth.groupby('bucket',as_index=False).agg(depth_bid_1pct=('depth_bid_1pct','mean'),depth_ask_1pct=('depth_ask_1pct','mean'),depth_imbalance_1pct=('depth_imbalance_1pct','mean')).rename(columns={'bucket':'ts'}); depth=depth[(depth.ts>=START)&(depth.ts<=END)]
print('Depth proxy rows:',len(depth))


In [ ]:
# BUILD CANONICAL 5m DATASET
f=fut_k.copy(); f['fut_delta_taker_base']=2*f.taker_buy_base-f.volume; f['fut_buy_ratio']=f.taker_buy_base/f.volume.replace(0,pd.NA); f['fut_ret_5m']=f.close.pct_change()
s=spot_k[['ts','close','volume','taker_buy_base']].rename(columns={'close':'spot_close','volume':'spot_volume','taker_buy_base':'spot_taker_buy_base'}); s['spot_delta_taker_base']=2*s.spot_taker_buy_base-s.spot_volume
df=f.merge(s,on='ts',how='left',validate='one_to_one').merge(flow,on='ts',how='left',validate='one_to_one').merge(metrics,on='ts',how='left',validate='one_to_one')
if not funding.empty: df=pd.merge_asof(df.sort_values('ts'),funding.sort_values('ts'),on='ts',direction='backward',tolerance=pd.Timedelta('8h'))
if MODE=='MICROSTRUCTURE' and not depth.empty: df=df.merge(depth,on='ts',how='left',validate='one_to_one')
if 'oi_btc' in df: df['oi_change_5m']=df.oi_btc.diff(); df['oi_change_1h']=df.oi_btc.diff(12); df['oi_change_4h']=df.oi_btc.diff(48); df['oi_pct_change_1h']=df.oi_btc.pct_change(12)
if 'taker_ls_volume_ratio' in df: rr=df.taker_ls_volume_ratio.rolling(48,min_periods=24); df['crowding_z_4h']=(df.taker_ls_volume_ratio-rr.mean())/rr.std()
rr=df.fut_delta_taker_base.rolling(48,min_periods=24); df['flow_z_4h']=(df.fut_delta_taker_base-rr.mean())/rr.std(); df['flow_accel_1h']=df.fut_delta_taker_base.diff(12)
df['rv_1h']=df.fut_ret_5m.rolling(12,min_periods=12).std(); df['rv_4h']=df.fut_ret_5m.rolling(48,min_periods=48).std(); den=df.fut_ret_5m.abs().rolling(12,min_periods=12).sum(); df['price_efficiency_1h']=df.fut_ret_5m.rolling(12,min_periods=12).sum().abs()/den.replace(0,pd.NA)
df['spot_futures_return_spread_5m']=df.spot_close.pct_change()-df.close.pct_change(); df['spot_futures_flow_sign_agree']=((df.fut_delta_taker_base>0)==(df.spot_delta_taker_base>0)).astype('int8')
df['fwd_ret_5m']=df.close.shift(-1)/df.close-1; df['fwd_ret_15m']=df.close.shift(-3)/df.close-1; df['fwd_ret_60m']=df.close.shift(-12)/df.close-1
df=df.sort_values('ts').drop_duplicates('ts').reset_index(drop=True); assert df.ts.is_monotonic_increasing and df.close.notna().all(); assert df.oi_btc.notna().sum()>0
CANON=OUT/'btcusdt_futures_5m_canonical.parquet'; df.to_parquet(CANON,index=False,compression='zstd'); print('CANONICAL:',CANON,'ROWS:',len(df),'COLS:',len(df.columns))


In [ ]:
# QUALITY AUDIT + EXPORT
u=df.ts.drop_duplicates().sort_values().reset_index(drop=True); gap=u.diff().dt.total_seconds().div(60); gaps=pd.DataFrame({'from_ts':u.shift(1),'to_ts':u,'gap_minutes':gap}).dropna(); gaps=gaps[gaps.gap_minutes>5]; gaps.to_csv(REPORTS/'gaps_gt_5m.csv',index=False)
expected=int((END.floor('5min')-START.floor('5min'))/pd.Timedelta('5min'))+1
quality={'generated_at_utc':datetime.now(timezone.utc).isoformat(),'mode':MODE,'symbol':SYMBOL,'requested_start':str(START),'requested_end':str(END),'rows':int(len(df)),'expected_5m_rows':expected,'coverage_ratio':float(len(df)/expected),'duplicate_timestamps':int(df.ts.duplicated().sum()),'gap_count_gt_5m':int(len(gaps)),'max_gap_minutes':float(gaps.gap_minutes.max()) if len(gaps) else 0.0,'oi_non_null':int(df.oi_btc.notna().sum()),'crowding_non_null':int(df.taker_ls_volume_ratio.notna().sum()) if 'taker_ls_volume_ratio' in df else 0,'agg_flow_non_null':int(df.agg_delta_usdt.notna().sum()),'funding_non_null':int(df.funding_rate.notna().sum()) if 'funding_rate' in df else 0,'spot_close_non_null':int(df.spot_close.notna().sum())}
(REPORTS/'quality.json').write_text(json.dumps(quality,indent=2)); (REPORTS/'kline_audit.json').write_text(json.dumps(fut_audit+spot_audit,indent=2,default=str)); (REPORTS/'flow_audit.json').write_text(json.dumps(flow_audit,indent=2,default=str)); (REPORTS/'metrics_audit.json').write_text(json.dumps(metric_audit,indent=2,default=str)); (REPORTS/'funding_audit.json').write_text(json.dumps([{'rows':len(funding)}],indent=2)); (REPORTS/'depth_audit.json').write_text(json.dumps(depth_audit,indent=2,default=str))
print('FINAL QUALITY'); print(json.dumps(quality,indent=2))
package=Path('/content/btcusdt_microstructure_research_FINAL.zip'); import zipfile
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(OUT))
print('PACKAGE:',package,'MB:',round(package.stat().st_size/1024/1024,2))
from google.colab import files; files.download(str(package))
